# Judge Self-Agreement Analysis

This notebook measures the self-agreement of the judge model. We re-run the judge on intervention pairs from the training data and measure how often it selects the same intervention.

In [ ]:
import json
from src import llms, utils, dataset_gen


## Load Training Data and Documents

In [ ]:
# Load the training data that contains original judge decisions
training_data_path = 'results/training_data.jsonl'

training_data = []
with open(training_data_path, 'r', encoding='utf-8') as f:
    for line in f:
        training_data.append(json.loads(line.strip()))

print(f"Loaded {len(training_data)} training examples")

Loaded 1637 training examples


In [ ]:
# Load the documents
documents = utils.load_dataset_from_jsonl('results/training_docs.jsonl')
print(f"Loaded {len(documents)} documents")

Loaded 100 documents


## Reconstruct Intervention Pairs for Re-judging

In [22]:
# Reconstruct intervention pairs from training data
# We need to convert the training data back to the format expected by generate_judge_dataset
intervention_pairs = []

for training_example in training_data:
    accepted = training_example['accepted']
    rejected = training_example['rejected']
    accepted_type = training_example['accepted_type']
    rejected_type = training_example['rejected_type']
    accepted_agent = training_example['accepted_agent']  # 1 or 2
    
    # Reconstruct original intervention_1 and intervention_2
    if accepted_agent == 1:
        intervention_1 = accepted
        intervention_1_type = accepted_type
        intervention_2 = rejected
        intervention_2_type = rejected_type
    elif accepted_agent == 2:
        intervention_1 = rejected
        intervention_1_type = rejected_type
        intervention_2 = accepted
        intervention_2_type = accepted_type
    else:
        raise ValueError(f"Invalid accepted_agent value: {accepted_agent}")
    
    intervention_pairs.append({
        'doc_index': training_example['doc_index'],
        'scenario': training_example['scenario'],
        'comments_used': training_example['comments_used'],
        'intervention_1': intervention_1,
        'intervention_1_type': intervention_1_type,
        'intervention_2': intervention_2,
        'intervention_2_type': intervention_2_type,
        'original_winner': accepted_agent  # Store original decision
    })

print(f"Reconstructed {len(intervention_pairs)} intervention pairs")

Reconstructed 1637 intervention pairs


## Calculate Self-Agreement

In [ ]:
# Initialize the same judge model used for generating training data
judge_llm = llms.OpenAiClient(model="gpt-4.1", temperature=0.4)

print(f"Using judge model: gpt-4.1 with temperature=0.4")

Using judge model: gpt-4.1 with temperature=0.4


In [ ]:
# Run judge again on all intervention pairs using parallel execution
print(f"Re-running judge on {len(intervention_pairs)} intervention pairs...")
print("This uses parallel execution for speed.\n")

# Sample a subset for testing (you can adjust or remove this limit)
sample_size = len(intervention_pairs)  # Use all examples, or set to smaller number for testing
intervention_pairs_sample = intervention_pairs[:sample_size]

# Use generate_judge_dataset to run judge in parallel
new_judge_results = dataset_gen.generate_judge_dataset(
    intervention_pairs=intervention_pairs_sample,
    documents=documents,
    judge_llm=judge_llm,
    judge_method='selection',
    skip_equal_rating=False,  # Don't skip ties, we want to measure everything
    agent_output_is_json=True
)

print(f"\nCompleted judging on {len(new_judge_results)} examples")

## Analyze Self-Agreement Results

In [27]:
# Create a mapping from (doc_index, scenario) to new judge result for easy lookup
new_results_map = {}
for result in new_judge_results:
    key = (result['doc_index'], result['scenario'])
    new_results_map[key] = result

# Compare original and new judge decisions
results = []
agreements = 0
disagreements = 0
ties_new = 0

for pair in intervention_pairs_sample:
    key = (pair['doc_index'], pair['scenario'])
    
    # Get new judge result
    if key not in new_results_map:
        print(f"Warning: No new result found for doc {key[0]}, scenario {key[1]}")
        continue
    
    new_result = new_results_map[key]
    original_winner = pair['original_winner']
    new_winner = new_result['accepted_agent']
    
    agreed = (original_winner == new_winner)
    
    result = {
        'doc_index': pair['doc_index'],
        'scenario': pair['scenario'],
        'original_winner': original_winner,
        'new_selection': new_winner,
        'agreed': agreed
    }
    results.append(result)
    
    # Count agreements and disagreements
    if agreed:
        agreements += 1
    else:
        disagreements += 1
    
    if new_winner == 0:
        ties_new += 1

# Calculate statistics
print(f"{'='*60}")
print(f"Self-Agreement Results")
print(f"{'='*60}")
print(f"Total examples tested: {len(results)}")
print(f"Agreements: {agreements} ({agreements/len(results)*100:.1f}%)")
print(f"Disagreements: {disagreements} ({disagreements/len(results)*100:.1f}%)")
print(f"New ties (judge selected 0 - both equal): {ties_new}")
print(f"\nSelf-agreement rate: {agreements/len(results)*100:.1f}%")

Self-Agreement Results
Total examples tested: 1637
Agreements: 1171 (71.5%)
Disagreements: 466 (28.5%)
New ties (judge selected 0 - both equal): 0

Self-agreement rate: 71.5%


## Analyze Disagreement Cases

In [ ]:
# Analyze disagreement cases
disagreement_cases = [r for r in results if not r['agreed']]

print(f"Found {len(disagreement_cases)} disagreement cases")
print(f"Disagreement rate: {len(disagreement_cases)/len(results)*100:.1f}%\n")

# Show breakdown of disagreements
if disagreement_cases:
    # Count different types of disagreements
    from collections import Counter
    disagreement_types = []
    
    for case in disagreement_cases:
        orig = case['original_winner']
        new = case['new_selection']
        disagreement_types.append(f"{orig} -> {new}")
    
    type_counts = Counter(disagreement_types)
    print("Disagreement breakdown:")
    for disagreement_type, count in type_counts.most_common():
        print(f"  {disagreement_type}: {count} cases ({count/len(disagreement_cases)*100:.1f}%)")
    
    # Show first few disagreement cases with details
    print(f"\nShowing first 5 disagreement cases:")
    for i, case in enumerate(disagreement_cases[:5]):
        print(f"\nDisagreement Case {i+1}:")
        print(f"  Doc Index: {case['doc_index']}, Scenario: {case['scenario']}")
        print(f"  Original winner: {case['original_winner']}")
        print(f"  New selection: {case['new_selection']}")